In [ ]:
# ==============================================================================
# PROJET           : AgroScan AI
# MODULE           : Pipeline de Prétraitement - Extracteur de Feuille
# SCRIPT           : Conversion U²-Netp -> TFLite (Float32 Uniquement via onnx2tf)
# DESCRIPTION
# ==============================================================================

# 0. Installation de l'outil de conversion moderne
!pip install -q onnx onnx2tf tensorflow

import os
import urllib.request
import shutil
from google.colab import files

# Configuration des chemins
onnx_url = "https://github.com/danielgatis/rembg/releases/download/v0.0.0/u2netp.onnx"
onnx_path = "u2netp.onnx"
final_tflite_name = "u2netp_float32.tflite"

print("📥 1. Téléchargement du modèle u2netp.onnx...")
if not os.path.exists(onnx_path):
    urllib.request.urlretrieve(onnx_url, onnx_path)
    print("✅ Téléchargement terminé.")
else:
    print("✅ Le fichier existe déjà.")

print("\n🔄 2. Conversion ONNX vers TFLite via onnx2tf...")
# CORRECTION : Utilisation de -i à la place de -in
!onnx2tf -i u2netp.onnx

# Localisation du fichier généré par onnx2tf
expected_path = os.path.join("saved_model", "u2netp_float32.tflite")

if os.path.exists(expected_path):
    # On déplace le fichier à la racine pour un accès plus propre
    if os.path.exists(final_tflite_name):
        os.remove(final_tflite_name)
    shutil.move(expected_path, final_tflite_name)
    print(f"\n🎉 Succès ! Modèle Float32 généré avec précision native : {final_tflite_name}")

    print("\n⬇️ 3. Téléchargement vers ton ordinateur...")
    try:
        files.download(final_tflite_name)
    except Exception as e:
        print(f"⚠️ Le téléchargement automatique a été bloqué par ton navigateur.")
        print(f"Tu peux récupérer '{final_tflite_name}' manuellement dans l'onglet 'Fichiers' (icône dossier à gauche de Colab).")
else:
    print("\n❌ Erreur : Le fichier .tflite n'a pas pu être localisé après la conversion.")

📥 1. Téléchargement du modèle u2netp.onnx...
✅ Le fichier existe déjà.

🔄 2. Conversion ONNX vers TFLite via onnx2tf...

Model optimizing started ============================================================
Simplifying...
Finish! Here is the difference:
┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃            ┃ Original Model ┃ Simplified Model ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ Add        │ 11             │ 11               │
│ Cast       │ 38             │ 0                │
│ Concat     │ 127            │ 51               │
│ Constant   │ 504            │ 250              │
│ Conv       │ 119            │ 119              │
│ Gather     │ 76             │ 0                │
│ MaxPool    │ 33             │ 33               │
│ Relu       │ 112            │ 112              │
│ Resize     │ 38             │ 38               │
│ Shape      │ 114            │ 0                │
│ Sigmoid    │ 7              │ 7                │
│ Slice      │ 38             │ 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from pathlib import Path

# Recherche récursive de tous les fichiers de poids (.pt) dans Colab
fichiers_poids = list(Path("/content").rglob("*.pt"))

print("--- Liste des modèles trouvés ---")
if fichiers_poids:
    for p in fichiers_poids:
        print(f"👉 {p}")
else:
    print("❌ Aucun fichier .pt trouvé sur la session.")

--- Liste des modèles trouvés ---
👉 /content/yolo26n.pt
👉 /content/yolov8n-seg.pt
👉 /content/runs/segment/runs_plant_seg/yolov8n_seg_320/weights/last.pt
👉 /content/runs/segment/runs_plant_seg/yolov8n_seg_320/weights/best.pt


In [ ]:
# ============================================================
# 4) Inférence de contrôle (Visualisation des masques)
# ============================================================
best_pt_path = Path("/content/runs/segment/runs_plant_seg/yolov8n_seg_320/weights/best.pt")

print(f"🔄 Chargement du modèle depuis : {best_pt_path}")
best_model = YOLO(str(best_pt_path))

# --- CORRECTION DU CHEMIN DE VALIDATION ---
# On récupère le chemin du YAML et on le force en absolu par rapport à DATASET_DIR
raw_val_path = data_cfg.get("val", "valid/images")
val_source = Path(DATASET_DIR / raw_val_path).resolve()

# Sécurité si le chemin résolu avec "../" sort du dossier ou est incorrect
if not val_source.exists():
    if (DATASET_DIR / "valid/images").exists():
        val_source = DATASET_DIR / "valid/images"
    elif (DATASET_DIR / "val/images").exists():
        val_source = DATASET_DIR / "val/images"

print(f"\n🔹 Inférence de contrôle sur le set de validation : {val_source}")

best_model.predict(
    source=str(val_source), # Utilisation du chemin absolu validé
    imgsz=IMG_SIZE,
    conf=0.25,
    save=True,
    project="runs_plant_seg",
    name=f"yolov8{MODEL_SIZE}_seg_320_preview"
)

# ============================================================
# 5) Export TFLite (Optimisé pour Android)
# ============================================================
print("\n🔹 Export TFLite float32...")
tflite_float_path = best_model.export(
    format="tflite",
    imgsz=IMG_SIZE,
    int8=False,
    dynamic=False
)

print("\n🔹 Export TFLite int8 (Quantized)...")
tflite_int8_path = best_model.export(
    format="tflite",
    imgsz=IMG_SIZE,
    int8=True,
    data=str(data_yaml_path),
    dynamic=False
)

print("\n🎯 Vos modèles de segmentation sont prêts !")
print("  • Modèle Précis (Float32) :", tflite_float_path)
print("  • Modèle Ultra-Léger (Int8) :", tflite_int8_path)

🔄 Chargement du modèle depuis : /content/runs/segment/runs_plant_seg/yolov8n_seg_320/weights/best.pt

🔹 Inférence de contrôle sur le set de validation : /content/datasets/Plant-Disease-Segmentation-1/valid/images

image 1/75 /content/datasets/Plant-Disease-Segmentation-1/valid/images/00b41556-fe22-41fb-93aa-76b0c45d923a___UF-GRC_YLCV_Lab-02644_JPG.rf.586f04497a9db48291f15e218d149f6d.jpg: 320x320 1 Tomato__Tomato_YellowLeaf__Curl_Virus, 17.3ms
image 2/75 /content/datasets/Plant-Disease-Segmentation-1/valid/images/00bce074-967b-4d50-967a-31fdaa35e688___RS_HL-0223_JPG.rf.7f8bfbe7ed2da8d5021840b1ecdc595f.jpg: 320x320 1 Tomato_healthy, 15.0ms
image 3/75 /content/datasets/Plant-Disease-Segmentation-1/valid/images/00c07a77-15e6-4815-92d4-8d1e1afb7f3c___PSU_CG-2052_JPG.rf.459f0aaa374d020aa531be0a4e794ab9.jpg: 320x320 1 Tomato__Tomato_mosaic_virus, 10.4ms
image 4/75 /content/datasets/Plant-Disease-Segmentation-1/valid/images/00c5c908-fc25-4710-a109-db143da23112___RS_Erly-B-7778_JPG.rf.6026d9c2b

In [ ]:
# ============================================================
# YOLOv8 Multi-Species Detection - CASCADE STAGE 1 (Vidéo Perso)
# ============================================================

!pip install -q ultralytics pyyaml opencv-python

import os
import yaml
import random
import shutil
import cv2
import xml.etree.ElementTree as ET
from pathlib import Path
from ultralytics import YOLO

# ─── 1. CONFIGURATION DES DOSSIERS ───
DATASET_DIR = Path("/content/leaf_detection_dataset")
for split in ["train", "val"]:
    for folder in ["images", "labels"]:
        (DATASET_DIR / split / folder).mkdir(parents=True, exist_ok=True)

# ─── 2. TÉLÉCHARGEMENT DATASET PLANTDOC ───
if not Path("/content/plantdoc_src").exists():
    print("Téléchargement du dataset source PlantDoc...")
    !git clone https://github.com/pratikkayal/PlantDoc-Object-Detection-Dataset.git /content/plantdoc_src

SPECIES_MAP = {
    "apple": 0, "blueberry": 1, "cherry": 2, "corn": 3, "grape": 4,
    "orange": 5, "peach": 6, "pepper": 7, "potato": 8, "raspberry": 9,
    "soybean": 10, "squash": 11, "strawberry": 12, "tomato": 13
}

# ─── 3. CONVERTISSEUR XML VERS YOLO ───
def convert_xml_to_yolo(xml_path, img_path, output_txt_path):
    img = cv2.imread(str(img_path))
    if img is None:
        return False

    h, w, _ = img.shape

    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
        yolo_lines = []

        for obj in root.findall('object'):
            raw_name = obj.find('name').text.lower()
            class_id = next((idx for keyword, idx in SPECIES_MAP.items() if keyword in raw_name), None)

            if class_id is None:
                continue

            xmlbox = obj.find('bndbox')
            xmin = max(0, min(float(xmlbox.find('xmin').text), w))
            ymin = max(0, min(float(xmlbox.find('ymin').text), h))
            xmax = max(0, min(float(xmlbox.find('xmax').text), w))
            ymax = max(0, min(float(xmlbox.find('ymax').text), h))

            if xmax <= xmin or ymax <= ymin:
                continue

            cx = ((xmin + xmax) / 2.0) / w
            cy = ((ymin + ymax) / 2.0) / h
            bw = (xmax - xmin) / w
            bh = (ymax - ymin) / h

            yolo_lines.append(f"{class_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")

        if yolo_lines:
            with open(output_txt_path, "w") as f:
                f.write("\n".join(yolo_lines))
            return True

    except ET.ParseError:
        print(f"Erreur de parsing XML : {xml_path}")
    except Exception as e:
        print(f"Erreur inattendue sur {xml_path}: {e}")

    return False

# ─── 4. RÉPARTITION DES DONNÉES PLANTDOC (85% TRAIN / 15% VAL) ───
all_pairs = []
for img_path in list(Path("/content/plantdoc_src").rglob("*.*")):
    if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png']:
        xml_path = img_path.with_suffix(".xml")
        if xml_path.exists():
            all_pairs.append((img_path, xml_path))

random.seed(42)
random.shuffle(all_pairs)
split_idx = int(0.85 * len(all_pairs))

def deploy_data(pairs, split):
    count = 0
    for img, xml in pairs:
        dst_img = DATASET_DIR / split / "images" / f"leaf_{split}_{count}{img.suffix}"
        dst_txt = DATASET_DIR / split / "labels" / f"leaf_{split}_{count}.txt"
        if convert_xml_to_yolo(xml, img, dst_txt):
            shutil.copy(img, dst_img)
            count += 1

print("Distribution des images de feuilles...")
deploy_data(all_pairs[:split_idx], "train")
deploy_data(all_pairs[split_idx:], "val")

# ─── 5. EXTRACTION ET INTÉGRATION DE LA VIDÉO DE MAINS (IMAGES NÉGATIVES) ───
VIDEO_PATH = "/content/video_main.mp4"
DOWNLOAD_DIR = Path("/content/hands_bg")
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

if not Path(VIDEO_PATH).exists():
    raise FileNotFoundError("❌ Le fichier 'video_main.mp4' est introuvable. Veuillez l'uploader dans le panneau de gauche de Colab avant de continuer.")

print("Extraction des images à partir de votre vidéo...")
cap = cv2.VideoCapture(VIDEO_PATH)

frame_skip = 10 # Extrait une image toutes les 10 frames pour éviter les doublons quasi identiques
count = 0
saved = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    if count % frame_skip == 0:
        img_path = DOWNLOAD_DIR / f"hand_frame_{saved}.jpg"
        cv2.mkdir = True
        cv2.imwrite(str(img_path), frame)
        saved += 1
    count += 1

cap.release()
print(f"✓ {saved} images réelles extraites de la vidéo.")

# Dispatch des images négatives dans le dataset YOLO
neg_images = list(DOWNLOAD_DIR.glob("*.jpg"))
random.shuffle(neg_images)

for i, img in enumerate(neg_images):
    split = "train" if i < int(len(neg_images) * 0.85) else "val"
    dst_name = f"bg_hand_video_{i}.jpg"

    # Copie du cadre de la vidéo
    shutil.copy(img, DATASET_DIR / split / "images" / dst_name)

    # CRUCIAL : Création du fichier de annotations vide pour marquer l'image comme "Background"
    (DATASET_DIR / split / "labels" / f"bg_hand_video_{i}.txt").touch()

print(f"✓ {len(neg_images)} images de fond (mains) intégrées avec succès au format Background.")

# ─── 6. ENTRAÎNEMENT SÉCURISÉ (SANS EXPLOSION DE RAM) ───
names_config = {
    0: "Apple", 1: "Blueberry", 2: "Cherry", 3: "Corn", 4: "Grape",
    5: "Orange", 6: "Peach", 7: "Pepper", 8: "Potato", 9: "Raspberry",
    10: "Soybean", 11: "Squash", 12: "Strawberry", 13: "Tomato"
}

yaml_path = DATASET_DIR / "data.yaml"
with open(yaml_path, "w") as f:
    yaml.safe_dump({
        "path": str(DATASET_DIR),
        "train": "train/images",
        "val": "val/images",
        "names": names_config
    }, f)

model = YOLO("yolov8n.pt")

# Lancement de l'entraînement
model.train(
    data=str(yaml_path),
    imgsz=640,
    epochs=150,
    patience=25,
    batch=16,
    device=0,
    workers=2,
    cache=False,            # Sécurité indispensable pour la RAM Colab
    project="leaf_detector",
    name="yolov8_species",
    mosaic=1.0,
    close_mosaic=15,
    hsv_s=0.5, hsv_v=0.4
)

# ─── 7. EXPORT DE L'IA FLUIDE POUR APPLICATION MOBILE ───
best_pt = list(Path("/content/leaf_detector").rglob("best.pt"))[0]
print(f"Exportation du modèle entraîné : {best_pt}")

YOLO(str(best_pt)).export(
    format="tflite",
    imgsz=640,
    int8=True,
    data=str(yaml_path)
)
print("🚀 Terminé ! Votre modèle TFLite final (INT8) est prêt dans vos dossiers.")

Distribution des images de feuilles...
Extraction des images à partir de votre vidéo...
✓ 73 images réelles extraites de la vidéo.
✓ 73 images de fond (mains) intégrées avec succès au format Background.
New https://pypi.org/project/ultralytics/8.4.62 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.61 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=15, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/leaf_detection_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.5, hsv_v=0.4

IndexError: list index out of range

In [ ]:
from pathlib import Path
from ultralytics import YOLO

print("Recherche du modèle entraîné...")

# On cherche dans le bon dossier créé par YOLO (/content/runs/detect/...)
best_pt_list = list(Path("/content/runs/detect/leaf_detector/yolov8_species-6").rglob("best.pt"))

if best_pt_list:
    best_pt = best_pt_list[0]
    print(f"✅ Modèle trouvé : {best_pt}")

    # Lancement de l'export TFLite
    print("⏳ Exportation en TFLite en cours (cela peut prendre 1 à 2 minutes)...")
    android_model = YOLO(str(best_pt))
    android_model.export(format="tflite", imgsz=320)

    print("🚀 Modèle TFLite généré avec succès !")
else:
    print("❌ Erreur : Le fichier best.pt est toujours introuvable. Vérifie les dossiers à gauche dans Colab.")

Recherche du modèle entraîné...
✅ Modèle trouvé : /content/runs/detect/leaf_detector/yolov8_species-6/weights/best.pt
⏳ Exportation en TFLite en cours (cela peut prendre 1 à 2 minutes)...
Ultralytics 8.4.61 🚀 Python-3.12.13 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
Model summary (fused): 73 layers, 3,008,378 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from '/content/runs/detect/leaf_detector/yolov8_species-6/weights/best.pt' with input shape (1, 3, 320, 320) BCHW and output shape(s) (1, 18, 2100) (5.9 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 12 packages in 328ms
Prepared 4 packages in 2.07s
Installed 4 packages in 415ms
 + colorama==0.4.6
 + onnx==1.21.0
 + onnxruntime==1.26.0
 + on